In [6]:
import pandas as pd
import numpy as np

## Загрузка данных

Загружаем исходный датасет с информацией о продажах товаров.

Предполагается, что в данных содержатся:
- идентификатор или название товара;
- период продажи (например, месяц);
- объем продаж.

Далее проверим структуру данных и убедимся, что они корректно загрузилис

In [2]:
df = pd.read_csv("/data1.csv", encoding = "1251")

df.head()

,DR_Dat,DR_Tim,DR_NChk,DR_NDoc,DR_Apt,DR_Kkm,DR_TDoc,DR_TPay,DR_CDrugs,DR_NDrugs,...,DR_Prod,DR_Kol,DR_CZak,DR_CRoz,DR_SDisc,DR_CDisc,DR_BCDisc,DR_TabEmpl,DR_VZak,DR_Pos
0,2022-08-01,08:06:18,1272,13002561,13,22589,Розничная реализация,18,144734,ГАСТАЛ №12 ТАБ. Д/РАСС.,...,TEVA Pharvaceutical Industries Ltd,1.0,196.71,270.0,0.0,NaN,NaN,29,1,1.0
1,2022-08-01,08:38:53,1273,13002561,13,22589,Розничная реализация,15,69661,"ТОБРОПТ 0,3% 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП. /РОМФАРМ/",...,РОМФАРМ КОМПАНИ ( ROMPHARM ),1.0,106.21,127.0,6.0,9.0,2.000100e+11,29,1,1.0
2,2022-08-01,08:55:38,1274,13002561,13,22589,Розничная реализация,18,190635,ЭЛИКВИС 5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-М...,...,Пфайзер,1.0,2320.99,2563.0,76.0,9.0,2.000100e+11,29,1,1.0
3,2022-08-01,09:00:40,1275,13002561,13,22589,Розничная реализация,18,276370,АРБИДОЛ МАКСИМУМ 200МГ. №10 КАПС. /ОТИСИФАРМ/Ф...,...,ОТИСИФАРМ ПАО,1.0,445.39,541.0,0.0,NaN,NaN,29,1,1.0
4,2022-08-01,09:04:05,1276,13002561,13,22589,Розничная реализация,15,2303,"ЭНАМ 2,5МГ. №20 ТАБ. /Д-Р РЕДДИ/",...,Д-р Редди с Лабораторис Лтд / Dr.REDDY's,1.0,18.04,22.0,1.0,9.0,2.000100e+11,29,1,5.0


## Подготовка данных для XYZ-анализа

XYZ-анализ основан на оценке **вариативности спроса** по каждому товару во времени.

Для этого необходимо:
1. сгруппировать продажи по товарам;
2. собрать значения продаж по периодам;
3. получить временной ряд продаж для каждого товара.

Именно на основе этих рядов далее будет рассчитан коэффициент вариации.

In [3]:
df.groupby(["DR_Dat", "DR_NDrugs"]).agg({"DR_Kol" : sum}).reset_index()

/tmp/ipykernel_15680/3835724152.py:1: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df.groupby(["DR_Dat", "DR_NDrugs"]).agg({"DR_Kol" : sum}).reset_index()


,DR_Dat,DR_NDrugs,DR_Kol
0,2022-08-01,"911-ВЕНОЛГОН ГЕЛЬ Д/НОГ ПРИ ТЯЖЕСТИ,БОЛИ,ОТЕКА...",1.0
1,2022-08-01,L-ТИРОКСИН 100МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,1.0
2,2022-08-01,L-ТИРОКСИН 50МКГ. №50 ТАБ. /БЕРЛИН ХЕМИ/,1.0
3,2022-08-01,L-ТИРОКСИН 75МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,2.0
4,2022-08-01,NF ВАТА ХИРУРГ. СТЕР. 100Г. /НЬЮФАРМ/,1.0
...,...,...,...
1084,2022-08-05,"ЭВО ПАНТЕНОЛ ПОМАДА ГИГИЕН. 2,8Г. [EVO] /АВАНТА/",1.0
1085,2022-08-05,ЭМОКСИПИН 10МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ. КРЫШ/КАП.,2.0
1086,2022-08-05,ЭСВИЦИН СР-ВО П/ОБЛЫСЕНИЯ ЛОСЬОН-ТОНИК 250МЛ. ...,1.0
1087,2022-08-05,ЭТОРИАКС 90МГ. №7 ТАБ. П/П/О,1.0


In [4]:
df1 = df.groupby("DR_NDrugs").agg({"DR_Dat" : "count"}).reset_index()

xyz_name = list(df1[df1["DR_Dat"] > 1]["DR_NDrugs"])

## Расчет показателей вариативности спроса

Для каждого товара рассчитываются следующие показатели:

- **среднее значение продаж** — показывает типичный уровень спроса;
- **стандартное отклонение** — отражает разброс значений относительно среднего;
- **коэффициент вариации** — основной показатель XYZ-анализа.

Чем ниже коэффициент вариации, тем стабильнее спрос на товар.

In [10]:
df = df.reset_index()
df = df[df["DR_NDrugs"].isin(xyz_name)][["DR_NDrugs", "DR_Kol"]]
df["cv"] = df.groupby("DR_NDrugs").apply(lambda x: x.std() / x.mean())

## Классификация товаров

После расчета коэффициента вариации товары распределяются по группам XYZ.

Обычно используются следующие пороговые значения:

- **X**: коэффициент вариации до 10%  
  Спрос стабильный, хорошо прогнозируется.
  
- **Y**: коэффициент вариации от 10% до 25%  
  Спрос подвержен умеренным колебаниям.
  
- **Z**: коэффициент вариации более 25%  
  Спрос нестабилен, прогнозирование затруднено.

Такая классификация позволяет определить, какие товары можно закупать и планировать более уверенно, а какие требуют осторожного подхода.

In [11]:
df["xyz"] = np.where(df["cv"] < 0.1, "X", np.where(df["cv"] < 0.25, "Y", "Z"))

df

,DR_NDrugs,DR_Kol,cv,xyz
0,ЭЛИКВИС 5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-М...,1.0,NaN,Z
1,АРБИДОЛ МАКСИМУМ 200МГ. №10 КАПС. /ОТИСИФАРМ/Ф...,1.0,NaN,Z
2,"ЭНАМ 2,5МГ. №20 ТАБ. /Д-Р РЕДДИ/",1.0,NaN,Z
3,"ЭНАМ 2,5МГ. №20 ТАБ. /Д-Р РЕДДИ/",1.0,NaN,Z
4,КЛОПИДОГРЕЛ-СЗ 75МГ. №28 ТАБ. П/П/О /СЕВЕРНАЯ ...,1.0,NaN,Z
...,...,...,...,...
900,КАЛЧЕК 5МГ. №30 ТАБ. /ИПКА/,1.0,NaN,Z
901,ТЕРАФЛЮ ЭКСТРА ЛИМОН ОТ ГРИППА И ПРОСТУДЫ 15Г....,0.2,NaN,Z
902,НО-ШПА 40МГ. №24 ТАБ. /ХИНОИН/,1.0,NaN,Z
903,АСКОРБИНКА ЛУНТИК ВИТ.С №10 ТАБ. (30Г.) /ФАРМ-...,1.0,NaN,Z
